# 04-visualization

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/mobility/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('mobility'))


In [ ]:
%pprint
import os
os.environ["KMP_DUPLICATE_LIB_OK"]  =  "TRUE"

In [ ]:
import numpy as np 
import pandas as pd
import os
import time
import datetime
import requests
import csv
import json
import re
import threading

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from tqdm import tqdm
import plotly.figure_factory as ff
import plotly.express as px

%config InlineBackend.figure_format = 'retina'
import warnings
# Keep warnings visible when checking the research environment.

# Global Function

In [ ]:
from analysis_utils import formalize_fip
def getCounties():
    # get a dictionary that maps a formalized fip (len == 5 str) to the corresponding county
    d = {}
    r = requests.get("https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt", timeout=30)
    r.raise_for_status()
    reader = csv.reader(r.text.splitlines(), delimiter=',')    
    for line in reader:
        d[line[1] + line[2]] = line[3].replace(" County","")
    
    extra_fip2county  = {'46102': 'Oglala Lakota', '02158': 'Kusilvak Census Area'}
    d.update(extra_fip2county)       
        
    return d
from analysis_utils import transform_mobilitydata
def compare_dtspp(fip_tuple, start_day='2020-03-01', end_day='2020-09-01', display_policy=False, plot_pic=False):
    fips = [formalize_fip(fip) for fip in fip_tuple]
    if display_policy:
        policies = poly if 'fips' in poly.columns else poly.reset_index()
        display(policies.loc[policies.fips.isin(fips)])
    result = transform_mobilitydata(dtspp.loc[dtspp.fips.isin(fips)],
                                   start_day=start_day, end_day=end_day)
    if plot_pic:
        result.plot(figsize=(15, 6))
    return result


# Import Data

In [ ]:
# Locate folder
os.chdir(workspace('mobility'))     

# policy
poly = pd.read_csv('Local-Policy-Responses-formalized-matched-localonly.csv')
poly.fips = poly.fips.apply(formalize_fip)
poly = poly.set_index('fips', drop=True)

# consumption
csp_byday = pd.read_csv('trends/google-trend-Coronavirus_disease_2019_byday_sorted.csv',index_col=0)    # compare by day
csp_byreg = pd.read_csv('trends/google-trend-Coronavirus_disease_2019_byregion_sorted.csv',index_col=0) # compare by region

# metro to county
metro2county = pd.read_csv('trends/googel-trend-metro2county-clean-version.csv',index_col=0)

# combined info
combined_info = pd.read_csv('combined_info_with_dl.csv')
combined_info.fips = combined_info.fips.apply(formalize_fip)
combined_info.date = pd.to_datetime(combined_info.date)

# combined info2
combined_info2 = pd.read_csv('combined_info.csv')
combined_info2.fips = combined_info2.fips.apply(formalize_fip)
combined_info2.date = pd.to_datetime(combined_info2.date)
dtspp = combined_info2
dtspp_trans = transform_mobilitydata(dtspp)


# Variation

In [ ]:
display(combined_info)
display(combined_info2)

## Plot Trend: M50

In [ ]:
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json', timeout=30) as response:
    counties = json.load(response)

In [ ]:
def feature_mean(start_day='2020-03-01', end_day='2020-12-01', col='m50', data=combined_info):
    
    # This function returns feature means within specified dates 
    # as a data frame that could be interpreted by the choropleth
    
    mean_mobility = data[(pd.to_datetime(start_day)<=data.date) & 
                    (data.date<=pd.to_datetime(end_day))][['fips',col]].groupby(by=['fips']).apply(lambda df: df[col].mean())
    df = pd.DataFrame(mean_mobility).rename(columns={0: 'Mean '+col}).reset_index()
    return df

In [ ]:
start_day = '2020-03-01'
end_day   = '2020-03-14'
df = feature_mean(start_day=start_day , end_day=end_day, col='m50', data=combined_info)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="Hot_r",
                    range_color=(0,30),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
start_day = '2020-03-15'
end_day   = '2020-05-31'
df = feature_mean(start_day=start_day , end_day=end_day, col='m50', data=combined_info)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="Hot_r",
                    range_color=(0,30),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
start_day = '2020-06-01'
end_day   = '2020-12-01'
df = feature_mean(start_day=start_day , end_day=end_day, col='m50', data=combined_info)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="Hot_r",
                    range_color=(0,30),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

## Plot Mobility Change According to Normal: DTSPP or M50_Pct_Change

In [ ]:
start_day = '2020-01-04'
end_day   = '2020-03-14'

df = feature_mean(start_day=start_day , end_day=end_day, col='dtspp', data=combined_info2)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="IceFire", color_continuous_midpoint=0,
                    range_color=(-66,120),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
start_day = '2020-03-15'
end_day   = '2020-05-31'
df = feature_mean(start_day=start_day , end_day=end_day, col='dtspp', data=combined_info2)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="IceFire", color_continuous_midpoint=0,
                    range_color=(-66,120),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
start_day = '2020-06-01'
end_day   = '2020-12-28'
df = feature_mean(start_day=start_day , end_day=end_day, col='dtspp', data=combined_info2)

fig = px.choropleth(df, geojson=counties, locations='fips', color=df.columns[1],
                    color_continuous_scale="IceFire", color_continuous_midpoint=0,
                    range_color=(-66,120),
                    scope="usa", title='Mobility by County '+start_day+' ~ '+end_day)

fig.update_layout(title_x=0.45, title_y=0.95, title_font_family="Times New Roman", title_font_size=20,
                  margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

# Metro-Wise Mobility: By mean

In [ ]:
metro2fip = metro2county.county_code.apply(lambda x: tuple(x.split(', ')))
metro_dtspp = pd.DataFrame(columns=metro2fip.index, index=dtspp_trans.index)

In [ ]:
%%time
for metro in tqdm(metro2fip.index):
    metro_dtspp[metro] = compare_dtspp(metro2fip[metro]).mean(axis=1)
metro_dtspp = metro_dtspp.dropna(axis=1)    

In [ ]:
metro_dtspp.plot(figsize=(20,10), legend=False);